In [33]:
#import libraries

import os.path as op
import pandas as pd
from pathlib import Path
import re
import sys

In [34]:
#define paths

annot_dir = Path("./dset/annotation-26-csvs")
out_dir = Path("./dset/derivatives/annotations")
out_dir.mkdir(parents=True, exist_ok=True)

In [35]:
def find_annotation_files(annotation_dir: Path):
    pattern = re.compile(r"^(S\d+E\d+R\d+)_(\d{3})\.csv$")
    groups = {}
    for p in sorted(annotation_dir.iterdir()):
        if not p.is_file():
            continue
        m = pattern.match(p.name)
        if not m:
            continue
        prefix = m.group(1)
        annot = m.group(2)
        groups.setdefault(prefix, []).append((annot, p))
    return groups

def detect_columns(df: pd.DataFrame):
    # detect index column
    if "index" in df.columns:
        idx_col = "index"
    else:
        # fallback: first column
        idx_col = df.columns[0]

    # detect valence and arousal columns (case-insensitive)
    val_cols = [c for c in df.columns if "valence" in c.lower()]
    aro_cols = [c for c in df.columns if "arousal" in c.lower()]

    val_col = val_cols[0] if val_cols else None
    aro_col = aro_cols[0] if aro_cols else None
    return idx_col, val_col, aro_col

def combine_group(prefix, files):
    # files: list of (annotator_code, Path)
    
    # Skip if less than 2 files
    if len(files) < 2:
        print(f"Skipping {prefix}: need at least 2 files, found {len(files)}")
        return None
    
    dfs = []
    renamed = []
    valid_count = 0  # Track valid files separately from loop index
    
    for annot, path in sorted(files):
        df = pd.read_csv(path)
        idx_col, val_col, aro_col = detect_columns(df)
        if val_col is None or aro_col is None:
            print(f"Skipping {path.name}: could not find valence/arousal columns", file=sys.stderr)
            continue

        # create minimal df using valid_count for consistent numbering
        valid_count += 1
        out = pd.DataFrame()
        out["index"] = df[idx_col]
        out[f"valence_{valid_count}"] = df[val_col].values
        out[f"arousal_{valid_count}"] = df[aro_col].values
        dfs.append(out)
        renamed.append((annot, f"valence_{valid_count}", f"arousal_{valid_count}"))

    if len(dfs) < 2:
        print(f"Skipping {prefix}: need at least 2 valid files after filtering, found {len(dfs)}")
        return None

    # merge on index
    merged = dfs[0]
    for d in dfs[1:]:
        merged = pd.merge(merged, d, on="index", how="outer")

    # sort by index if numeric
    try:
        merged["index"] = pd.to_numeric(merged["index"])
        merged = merged.sort_values("index").reset_index(drop=True)
    except Exception:
        merged = merged.reset_index(drop=True)

    # Reorder columns: index, all valence_*, then all arousal_*
    val_cols = [c for c in merged.columns if c.startswith("valence_")]
    aro_cols = [c for c in merged.columns if c.startswith("arousal_")]

    def _num_suffix(colname):
        parts = colname.split("_")
        try:
            return int(parts[-1])
        except Exception:
            return 0

    val_cols = sorted(val_cols, key=_num_suffix)
    aro_cols = sorted(aro_cols, key=_num_suffix)

    new_cols = ["index"] + val_cols + aro_cols
    tail = [c for c in merged.columns if c not in new_cols]
    merged = merged[new_cols + tail]

    # Remove rows with insufficient ratings (fewer than 2 non-null raters)
    initial_rows = len(merged)
    removed_clips = []
    
    episode_match = re.match(r"(S\d+E\d+)R(\d+)", prefix)
    episode_id = episode_match.group(1) if episode_match else "Unknown"
    run_number = int(episode_match.group(2)) if episode_match else 0
    
    rows_to_keep = []
    for i, row in merged.iterrows():
        try:
            if isinstance(row["index"], str) and "_clip" in row["index"]:
                clip_number = int(row["index"].split("_clip")[-1])
            else:
                clip_number = i + 1
        except (ValueError, IndexError):
            clip_number = i + 1
        
        val_ratings = [float(row[col]) for col in val_cols if pd.notna(row[col])]
        aro_ratings = [float(row[col]) for col in aro_cols if pd.notna(row[col])]
        
        removal_reasons = []
        if len(val_ratings) < 2:
            removal_reasons.append(f"insufficient valence ratings ({len(val_ratings)})")
        if len(aro_ratings) < 2:
            removal_reasons.append(f"insufficient arousal ratings ({len(aro_ratings)})")
        
        if len(val_ratings) >= 2 and len(aro_ratings) >= 2:
            rows_to_keep.append(i)
        else:
            removed_clips.append({
                "episode": episode_id,
                "run": run_number,
                "run_prefix": prefix,
                "index": row["index"],
                "clip_number": clip_number,
                "valence_ratings": len(val_ratings),
                "arousal_ratings": len(aro_ratings),
                "reason": "; ".join(removal_reasons)
            })
    
    merged_filtered = merged.iloc[rows_to_keep].reset_index(drop=True) if rows_to_keep else pd.DataFrame(columns=merged.columns)
    
    if removed_clips:
        print(f"  Removed {len(removed_clips)}/{initial_rows} clips with insufficient ratings:")
        for clip in removed_clips[:5]:
            print(f"    - {clip['index']}: {clip['reason']}")
        if len(removed_clips) > 5:
            print(f"    ... and {len(removed_clips) - 5} more")
    
    return merged_filtered, renamed, removed_clips

In [36]:
groups = find_annotation_files(annot_dir)

In [37]:
groups

{'S01E01R01': [('002', PosixPath('dset/annotation-26-csvs/S01E01R01_002.csv')),
  ('003', PosixPath('dset/annotation-26-csvs/S01E01R01_003.csv')),
  ('005', PosixPath('dset/annotation-26-csvs/S01E01R01_005.csv')),
  ('008', PosixPath('dset/annotation-26-csvs/S01E01R01_008.csv'))],
 'S01E01R02': [('001', PosixPath('dset/annotation-26-csvs/S01E01R02_001.csv')),
  ('002', PosixPath('dset/annotation-26-csvs/S01E01R02_002.csv')),
  ('003', PosixPath('dset/annotation-26-csvs/S01E01R02_003.csv')),
  ('004', PosixPath('dset/annotation-26-csvs/S01E01R02_004.csv')),
  ('008', PosixPath('dset/annotation-26-csvs/S01E01R02_008.csv'))],
 'S01E01R03': [('004', PosixPath('dset/annotation-26-csvs/S01E01R03_004.csv')),
  ('005', PosixPath('dset/annotation-26-csvs/S01E01R03_005.csv')),
  ('006', PosixPath('dset/annotation-26-csvs/S01E01R03_006.csv'))],
 'S01E01R04': [('001', PosixPath('dset/annotation-26-csvs/S01E01R04_001.csv')),
  ('005', PosixPath('dset/annotation-26-csvs/S01E01R04_005.csv')),
  ('008

In [39]:
# Process all files but organize by episode instead of saving individual run files
episode_data = {}
all_removals = []

for prefix, files in groups.items():
    out = combine_group(prefix, files)
    if out is None:
        print(f"No valid files for {prefix}")
        continue
    
    merged, renamed, removed_clips = out
    
    # Extract episode from prefix (e.g., S01E01R01 -> S01E01)
    episode_match = re.match(r"(S\d+E\d+)R\d+", prefix)
    if episode_match:
        episode = episode_match.group(1)
        
        # Add run identifier column
        merged['run'] = prefix
        
        # Store in episode data structure
        if episode not in episode_data:
            episode_data[episode] = []
        episode_data[episode].append((prefix, merged, len(files), len(removed_clips)))
    
    # Track removals for documentation
    if removed_clips:
        all_removals.extend(removed_clips)
    
    print(f"Processed {prefix} ({len(files)} files -> {len(merged)} rows, {len(removed_clips) if removed_clips else 0} clips removed)")

# Create episode-level CSV files directly
print(f"\nCreating episode-level files for {len(episode_data)} episodes...")
episode_summary = []

for episode, run_data_list in episode_data.items():
    print(f"\nCombining runs for {episode}:")
    
    episode_dfs = []
    for run_prefix, run_df, file_count, removed_count in sorted(run_data_list):
        episode_dfs.append(run_df)
        print(f"  Added {run_prefix}: {len(run_df)} clips ({file_count} files, {removed_count} clips removed)")
    
    if episode_dfs:
        # Concatenate all runs for this episode
        episode_combined = pd.concat(episode_dfs, ignore_index=True)
        
        # Reorder columns to put run first, then index, then emotions
        val_cols = [c for c in episode_combined.columns if c.startswith("valence_")]
        aro_cols = [c for c in episode_combined.columns if c.startswith("arousal_")]
        other_cols = [c for c in episode_combined.columns if c not in ['run', 'index'] + val_cols + aro_cols]
        
        new_order = ['run', 'index'] + val_cols + aro_cols + other_cols
        episode_combined = episode_combined[new_order]
        
        # Save episode-level file
        episode_path = out_dir / f"{episode}.csv"
        episode_combined.to_csv(episode_path, index=False)
        
        total_clips = len(episode_combined)
        episode_summary.append((episode, len(run_data_list), episode_path, total_clips))
        print(f"  Saved {episode_path}: {total_clips} total clips across {len(run_data_list)} runs")

print(f"\nEpisode-level files created:")
for episode, run_count, file_path, clip_count in episode_summary:
    print(f"  {episode}: {run_count} runs, {clip_count} clips -> {file_path}")

print(f"\nTotal clips removed across all episodes: {len(all_removals)}")

# Save removal documentation
if all_removals:
    removal_df = pd.DataFrame(all_removals)
    
    # Debug: Check what columns we actually have
    print(f"\nDEBUG: Removal dataframe columns: {list(removal_df.columns)}")
    print(f"DEBUG: First few rows of removal data:")
    print(removal_df.head())
    
    # Calculate sequential position within each episode
    removal_df_with_positions = []
    
    for episode in removal_df['episode'].unique():
        print(f"\nProcessing episode positions for {episode}...")
        episode_removals = removal_df[removal_df['episode'] == episode].copy()
        
        # Build complete episode clip sequence from annotation files
        # We need to get the CORRECT sequential order across runs
        episode_prefixes = [prefix for prefix in groups.keys() if prefix.startswith(episode)]
        print(f"  Found prefixes for {episode}: {sorted(episode_prefixes)}")
        
        # Build cumulative episode position mapping based on actual run sequence
        episode_clip_positions = {}
        cumulative_position = 1  # Start from 1 for episode positions
        
        for run_prefix in sorted(episode_prefixes):
            files = groups[run_prefix]
            # Use the first available file for this run
            if not files:
                print(f"    {run_prefix}: No annotation files found!")
                continue
            ref_file = files[0][1]
            file_type = files[0][0]
            
            df = pd.read_csv(ref_file)
            
            # For this run, assign sequential positions
            run_clips = df['index'].tolist()
            print(f"    {run_prefix}: Processing {len(run_clips)} clips using {file_type} file, starting at position {cumulative_position}")
            
            for clip_index in run_clips:
                episode_clip_positions[clip_index] = cumulative_position
                cumulative_position += 1
                
            print(f"    {run_prefix}: Clips {run_clips[0]} to {run_clips[-1]} -> positions {episode_clip_positions[run_clips[0]]} to {episode_clip_positions[run_clips[-1]]}")
        
        print(f"  Total clips for {episode}: {len(episode_clip_positions)}")
        print(f"  Episode positions range: 1 to {cumulative_position - 1}")
        
        # Sample of the mapping for verification
        sample_clips = list(episode_clip_positions.items())[:3]
        print(f"  Sample position mapping: {sample_clips}")
        
        # Add positions to removals
        if 'index' in episode_removals.columns:
            episode_removals['episode_position'] = episode_removals['index'].map(episode_clip_positions)
            
            # Check mapping success
            mapped_count = episode_removals['episode_position'].notna().sum()
            total_count = len(episode_removals)
            print(f"  Mapped {mapped_count}/{total_count} positions for {episode}")
            
            # Fill any NaN positions with fallback logic
            nan_mask = episode_removals['episode_position'].isna()
            if nan_mask.any():
                print(f"  WARNING: {nan_mask.sum()} clips in {episode} could not be mapped")
                
                # For clips that couldn't be mapped, use clip number as fallback
                for idx, row in episode_removals[nan_mask].iterrows():
                    if pd.isna(row['episode_position']):
                        # Try to use clip_number directly as fallback
                        if pd.notna(row['clip_number']):
                            episode_removals.loc[idx, 'episode_position'] = float(row['clip_number'])
                            print(f"    Used clip_number {row['clip_number']} as fallback position for {row['index']}")
                        else:
                            print(f"    Could not determine position for {row['index']}")
        else:
            print(f"  WARNING: 'index' column not found in episode_removals for {episode}")
            print(f"  Available columns: {list(episode_removals.columns)}")
            # Use clip_number as fallback
            episode_removals['episode_position'] = episode_removals['clip_number']
        
        removal_df_with_positions.append(episode_removals)
    
    # Combine all episodes
    final_removal_df = pd.concat(removal_df_with_positions, ignore_index=True)
    
    # Ensure episode_position is preserved and properly formatted
    if 'episode_position' in final_removal_df.columns:
        # Convert to numeric where possible, keeping NaN as NaN
        final_removal_df['episode_position'] = pd.to_numeric(final_removal_df['episode_position'], errors='coerce')
        
        # Final check for any remaining NaN positions and use clip_number as ultimate fallback
        remaining_nan = final_removal_df['episode_position'].isna()
        if remaining_nan.any():
            print(f"\nFinal fallback: Using clip_number for {remaining_nan.sum()} remaining NaN positions")
            final_removal_df.loc[remaining_nan, 'episode_position'] = final_removal_df.loc[remaining_nan, 'clip_number']
    else:
        print("ERROR: episode_position column missing from final dataframe!")
        print(f"Available columns: {list(final_removal_df.columns)}")
    
    # Save removal log
    removal_path = out_dir / "removed_clips_log.csv"
    final_removal_df.to_csv(removal_path, index=False)
    print(f"\nSaved removal log: {removal_path}")
    print(f"Removal log contains episode_position column: {'episode_position' in final_removal_df.columns}")
    
    # Verify episode_position values
    if 'episode_position' in final_removal_df.columns:
        non_null_positions = final_removal_df['episode_position'].notna().sum()
        total_removals = len(final_removal_df)
        print(f"Episode positions mapped: {non_null_positions}/{total_removals} ({non_null_positions/total_removals*100:.1f}%)")
        
        # Show sample of episode positions by episode
        print(f"\nSample episode positions:")
        for episode in sorted(final_removal_df['episode'].unique()):
            episode_data = final_removal_df[final_removal_df['episode'] == episode]
            sample_positions = episode_data[['index', 'clip_number', 'episode_position']].head(3)
            print(f"  {episode}:")
            for _, row in sample_positions.iterrows():
                print(f"    {row['index']} (clip {row['clip_number']}) -> episode position {row['episode_position']}")
    
    # Print summary by episode
    print(f"\nRemovals by episode:")
    for episode in sorted(removal_df['episode'].unique()):
        episode_removals = final_removal_df[final_removal_df['episode'] == episode]
        mapped_positions = episode_removals['episode_position'].notna().sum() if 'episode_position' in final_removal_df.columns else 0
        print(f"  {episode}: {len(episode_removals)} clips removed, {mapped_positions} with positions")
        
else:
    print("No clips were removed during processing.")

Processed S01E01R01 (4 files -> 334 rows, 0 clips removed)
Processed S01E01R02 (5 files -> 276 rows, 0 clips removed)
Processed S01E01R03 (3 files -> 275 rows, 0 clips removed)
Processed S01E01R04 (3 files -> 286 rows, 0 clips removed)
Processed S01E01R05 (4 files -> 312 rows, 0 clips removed)
Processed S01E01R06 (4 files -> 333 rows, 0 clips removed)
Processed S01E02R01 (3 files -> 313 rows, 0 clips removed)
Processed S01E02R02 (3 files -> 250 rows, 0 clips removed)
Processed S01E02R03 (3 files -> 362 rows, 0 clips removed)
Processed S01E02R04 (3 files -> 354 rows, 0 clips removed)
Processed S01E02R05 (3 files -> 295 rows, 0 clips removed)
Processed S01E02R06 (4 files -> 218 rows, 0 clips removed)
Processed S01E02R07 (4 files -> 294 rows, 0 clips removed)
Processed S01E03R01 (4 files -> 272 rows, 0 clips removed)
Processed S01E03R02 (3 files -> 311 rows, 0 clips removed)
  Removed 1/317 clips with insufficient ratings:
    - S01E03R03_clip0017: insufficient valence ratings (1)
Process